# Module 1.1 — Data, Batching & the Training Loop
**DeskMate SLM Curriculum · Phase 1 · Notebook 02**

Run top-to-bottom on **Google Colab** (free T4) or **Kaggle**.

By the end you will have:
- A character-level tokenizer encoding a real text corpus
- A working `get_batch` function and a confirmed understanding of the shift-by-one convention
- A trained bigram language model
- A loss curve (train + val) and generated text before/after training

Read `1.1_bigram.md` first for the theory behind each step.

---


## Step 0 — Install & Seed

In [ ]:
%%capture
!pip install -q torch==2.3.1 matplotlib==3.9.0


In [ ]:
import random, os, pathlib, math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# ── Device ─────────────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device}")
if device == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")


## Step 1 — Runtime & Paths

In [ ]:
try:
    import google.colab
    RUNTIME = "colab"
    PROJECT_ROOT = pathlib.Path("/content/slm")
except ImportError:
    try:
        import kaggle_secrets
        RUNTIME = "kaggle"
        PROJECT_ROOT = pathlib.Path("/kaggle/working/slm")
    except ImportError:
        RUNTIME = "local"
        PROJECT_ROOT = pathlib.Path(".").resolve()

DATA_DIR   = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Runtime : {RUNTIME}")
print(f"Data    : {DATA_DIR}")
print(f"Models  : {MODELS_DIR}")


## Step 2 — Corpus: Tiny Shakespeare

We use Tiny Shakespeare (~1 MB) — a classic benchmark for character-level LMs.
Small enough to train on CPU in minutes; rich enough to see recognisable text emerge.


In [ ]:
corpus_path = DATA_DIR / "tiny_shakespeare.txt"

if not corpus_path.exists():
    import urllib.request
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    print(f"Downloading corpus from {url} ...")
    urllib.request.urlretrieve(url, corpus_path)
    print("Done.")

text = corpus_path.read_text(encoding="utf-8")
print(f"Corpus size    : {len(text):,} characters")
print(f"First 200 chars:\n{text[:200]}")


## Step 3 — Character Tokenizer

Build a vocabulary from every unique character in the corpus.
Encode the full text as a 1-D LongTensor, then split 90/10 train/val.


In [ ]:
# Vocabulary
chars = sorted(set(text))
V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

print(f"Vocabulary size : {V}")
print(f"Characters      : {''.join(chars)!r}")

# Sample round-trip
sample = "To be, or not to be"
print(f"\nEncode {sample!r}")
print(f"  → {encode(sample)}")
print(f"  → {decode(encode(sample))!r}")


In [ ]:
# Encode full corpus
data = torch.tensor(encode(text), dtype=torch.long)
print(f"data shape : {data.shape}   dtype : {data.dtype}")

# Train / val split (position-based, NOT shuffled)
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]
print(f"Train tokens : {len(train_data):,}")
print(f"Val tokens   : {len(val_data):,}")


## Step 4 — `get_batch`: Random Context/Target Sampling

Each call samples `B` random starting positions and returns `(x, y)` tensors of shape `(B, T)`.
`y` is `x` shifted one position to the right — the shift-by-one convention.


In [ ]:
B = 32   # batch size
T = 64   # context length (block size)

def get_batch(split):
    src = train_data if split == "train" else val_data
    ix = torch.randint(len(src) - T, (B,))
    x  = torch.stack([src[i   : i+T  ] for i in ix])   # (B, T)
    y  = torch.stack([src[i+1 : i+T+1] for i in ix])   # (B, T)
    return x.to(device), y.to(device)

# Inspect one batch
xb, yb = get_batch("train")
print(f"x shape : {xb.shape}  (B={B}, T={T})")
print(f"y shape : {yb.shape}")

print("\nShift-by-one check (first batch item, first 8 positions):")
print(f"  x[0, :8] = {xb[0, :8].tolist()}")
print(f"  y[0, :8] = {yb[0, :8].tolist()}")
print(f"  x decoded: {decode(xb[0, :8].tolist())!r}")
print(f"  y decoded: {decode(yb[0, :8].tolist())!r}")
print("  → y is x shifted one step right — model learns to predict each next character.")


In [ ]:
# Show the T training pairs packed into one sequence
print("Training signal packed in one sequence of length T:")
print(f"{'t':>3}  {'context':<20}  {'target':>6}")
print("-" * 35)
for t in range(min(8, T)):
    ctx = decode(xb[0, :t+1].tolist())
    tgt = itos[yb[0, t].item()]
    print(f"{t:>3}  {ctx!r:<20}  {tgt!r:>6}")


## Step 5 — Bigram Language Model

The simplest possible language model: one embedding table of shape `(V, V)`.
Row `i` is the logit distribution over all next tokens given current token `i`.
No attention, no history — just a learned lookup.


In [ ]:
class BigramLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Each token ID maps to a row of logits over the vocabulary
        self.embed = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx):
        # idx: (B, T)  →  logits: (B, T, V)
        return self.embed(idx)

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        # idx: (B, 1) starting context
        for _ in range(max_new_tokens):
            logits = self(idx)               # (B, T, V)
            logits = logits[:, -1, :]        # last position only → (B, V)
            probs  = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat([idx, next_id], dim=1)             # grow sequence
        return idx


model = BigramLM(V).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters : {n_params:,}  ({n_params/1e3:.1f}k)")

# Generate BEFORE training — should look like random noise
torch.manual_seed(SEED)
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_before = decode(model.generate(context, 200)[0].tolist())
print(f"\nGenerated (before training):\n{generated_before}")


## Step 6 — Training Loop

`forward → loss → backward → step`. We also evaluate on the val set every
`eval_interval` steps using `@torch.no_grad()` to avoid accumulating gradients.


In [ ]:
@torch.no_grad()
def estimate_loss(eval_iters=100):
    model.eval()
    results = {}
    for split in ("train", "val"):
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            logits = model(xb)
            losses[k] = F.cross_entropy(logits.view(-1, V), yb.view(-1))
        results[split] = losses.mean().item()
    model.train()
    return results


In [ ]:
# Hyperparameters
MAX_STEPS     = 5_000
LR            = 1e-2
EVAL_INTERVAL = 500

opt = torch.optim.AdamW(model.parameters(), lr=LR)

train_losses, val_losses, steps_logged = [], [], []

# Baseline loss before any training
init = estimate_loss()
print(f"Initial  →  train: {init['train']:.4f}  val: {init['val']:.4f}")
print(f"          (random baseline ≈ log({V}) = {math.log(V):.4f})")


In [ ]:
torch.manual_seed(SEED)

for step in range(MAX_STEPS):
    xb, yb = get_batch("train")

    logits = model(xb)                                        # (B, T, V)
    loss   = F.cross_entropy(logits.view(-1, V), yb.view(-1))

    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()

    if step % EVAL_INTERVAL == 0 or step == MAX_STEPS - 1:
        est = estimate_loss()
        train_losses.append(est["train"])
        val_losses.append(est["val"])
        steps_logged.append(step)
        print(f"step {step:5d}  train: {est['train']:.4f}  val: {est['val']:.4f}")

print("\nTraining complete.")


## Loss Curve

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(steps_logged, train_losses, label="train loss", color="#4C72B0")
plt.plot(steps_logged, val_losses,   label="val loss",   color="#DD8452", linestyle="--")
plt.axhline(math.log(V), color="grey", linestyle=":", label=f"random baseline (log {V} ≈ {math.log(V):.2f})")
plt.xlabel("Step")
plt.ylabel("Cross-entropy loss")
plt.title("Bigram LM — Training & Validation Loss")
plt.legend()
plt.tight_layout()
plt.savefig(str(MODELS_DIR / "bigram_loss_curve.png"), bbox_inches="tight")
plt.show()
print(f"Saved to {MODELS_DIR / 'bigram_loss_curve.png'}")


## Generated Text After Training

In [ ]:
torch.manual_seed(SEED)
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_after = decode(model.generate(context, 400)[0].tolist())

print("=== Before training ===")
print(generated_before[:200])
print()
print("=== After training (5 000 steps) ===")
print(generated_after)


In [ ]:
# Save the trained model weights
ckpt_path = MODELS_DIR / "bigram_ckpt.pt"
torch.save(model.state_dict(), ckpt_path)
print(f"Checkpoint saved: {ckpt_path}  ({ckpt_path.stat().st_size / 1024:.1f} KB)")


## Checkpoint

> **What exactly is the model predicting at each position, and what is the loss measuring?**

Write your answer in the cell below before moving on.

Hints:
- What is `y[b, t]` relative to `x[b, t]`?
- What does `logits[b, t, :]` represent?
- If the model assigned probability 1.0 to the correct token at every position, what would the loss be?
- If the model were random (uniform over V), what would the loss be? (Check against the baseline line on your chart.)


In [ ]:
answer = """
[Write your answer here]
"""
print(answer)


## Deliverable Summary

| Artifact | Location |
|---|---|
| Trained bigram weights | `models/bigram_ckpt.pt` |
| Loss curve | `models/bigram_loss_curve.png` |

**What you've locked in:** the `get_batch → forward → loss → backward → step → estimate_loss` skeleton. This exact structure carries forward through every module in Phase 1. In Module 1.2 you'll replace `BigramLM` with a model that includes a real attention head — but the loop around it does not change.

**Next:** Module 1.2 — Self-attention, by hand.
